## This notebook explores colors using the different flux measurements

In the Rubin catalogs, several flux measurements are reported for each object.   This notebook explores some of those, 
and the relationships between them.

The types of fluxes we look at here are:

psf:      Flux in a point-spread-function weighted aperture.

gaap1p0:  Flux in a 1.0 arcsec, point-spread-function weighted aperture

gaap3p0:  Flux in a 3.0 arcsec, point-spread-function weighted aperture

cModel:   Flux as computed in a model fit (for the entire object)

sersic:   Flux as computed by a fit to a sersic profile (for the entire object)


We are going to apply a few quality cuts, convert the fluxes to magnitudes and then to colors


In [ ]:
import os
import tables_io
import numpy as np
import matplotlib.pyplot as plt
import utility_functions
import plotting_functions

#### Download and uncompress the data

In [ ]:
_ = utility_functions.setup_data()

#### Change this to match the correct location

In [ ]:
pz_dir = '.'

#### Read the DP1 Object data 

This a sample of data from one "Tract", which is about a 1 degree x 1 degree area on the sky.

This particular tract is the ECDFS field for Rubin DP1.   It was position information removed.  (This was done to comply with data embargo issues at the time I made the file.)

In [ ]:
t = tables_io.read(f'{pz_dir}/data/object.hdf5')

You might want to explore a bit to see that is in the data that we just read

In [ ]:
print(f"Dataset has {len(t.keys())} columns and {len(t['g_psfFlux'])} objects")

#### Define data quality mask

Here we just require that the object by exteneded (i.e., not a point-like star) and that the 
signal-to-noise in the g,r and i bands all be > 10

In [ ]:
# Quality mask: keep extended objects (galaxies, not stars) with S/N > 10 in g, r, i.
# i_extendedness == 1 means the object is resolved (galaxy-like).
# S/N cuts remove objects with unreliable photometry where noise would dominate the colors.
mask  = np.bitwise_and(
    np.bitwise_and(
        t['i_extendedness'] == 1,
        t['g_psfFlux'] / t['g_psfFluxErr'] > 10,
    ),
    np.bitwise_and(
        t['i_psfFlux'] / t['i_psfFluxErr'] > 10,
        t['r_psfFlux'] / t['r_psfFluxErr'] > 10,
    ),
)

#### Get the magnitudes

This functions grabs the fluxes from the table we read in, and convert them to magnitudes

In [ ]:
# Extract five different flux measurements for each object and convert to magnitudes.
# PSF: optimal for point sources (stars), but misses extended galaxy flux.
# cModel: fits the object as a composite exponential + de Vaucouleurs profile → best total flux.
# Sersic: fits a single Sersic n profile; most flexible but noisiest.
# gaap1p0 / gaap3p0: fixed 1.0 / 3.0 arcsec Gaussian apertures, useful for comparing colors
#   across objects since the aperture is the same physical angular size.

# The make_mag_dict function extracts fluxes for the object table
# converts them to magnitudes and returns a dictionary key by band name
mags_psf = utility_functions.make_mag_dict(t, '{band}_psfFlux', 'ugrizy')
mags_cModel = utility_functions.make_mag_dict(t, '{band}_cModelFlux', 'ugrizy')
mags_sersic = utility_functions.make_mag_dict(t, '{band}_sersicFlux', 'ugrizy')
mags_1p0 = utility_functions.make_mag_dict(t, '{band}_gaap1p0Flux', 'ugrizy')
mags_3p0 = utility_functions.make_mag_dict(t, '{band}_gaap3p0Flux', 'ugrizy')

You might want to explore a bit to see what we just extracted.  Note that there are many np.nan, which are used
for non-detections.

## Make some plots 

#### Start by comparing the cModel to other fluxes

plot_mag_mag_scatter puts the first argument on the x-axis and the second argument on the y axis

plot_mag_mag_resid plots the first argument - the second argument against the second argument

In [ ]:
plotting_functions.plot_mag_mag_scatter(mags_cModel, mags_psf, 'ugrizy')
plotting_functions.plot_mag_mag_resid(mags_cModel, mags_psf, 'ugrizy')

## Let's stop and think a bit about what we are seeing and why that might be

#### Ok, now let's compare the c-model fluxes to the sersic profile fluxes

In [ ]:
plotting_functions.plot_mag_mag_scatter(mags_cModel, mags_sersic, 'ugrizy')
plotting_functions.plot_mag_mag_resid(mags_cModel, mags_sersic, 'ugrizy')

In [ ]:
plotting_functions.plot_mag_mag_scatter(mags_cModel, mags_1p0, 'ugrizy', mask=mask)
plotting_functions.plot_mag_mag_resid(mags_cModel, mags_1p0, 'ugrizy', mask=mask)

#### Again, let's stop and think a bit about what we are seeing here.

#### Now compare the psf and gaap_1p0 mags

In [ ]:
plotting_functions.plot_mag_mag_scatter(mags_1p0, mags_psf, 'ugrizy', mask=mask)
plotting_functions.plot_mag_mag_resid(mags_1p0, mags_psf, 'ugrizy', mask=mask)

#### Now we extract colors

In [ ]:
# Extract colors (adjacent-band magnitude differences) for each flux type.
colors_psf = utility_functions.make_color_dict(t, '{band}_psfFlux', 'ugrizy')
colors_cModel = utility_functions.make_color_dict(t, '{band}_cModelFlux', 'ugrizy')
colors_1p0 = utility_functions.make_color_dict(t, '{band}_gaap1p0Flux', 'ugrizy')
colors_3p0 = utility_functions.make_color_dict(t, '{band}_gaap3p0Flux', 'ugrizy')
colors_sersic = utility_functions.make_color_dict(t, '{band}_sersicFlux', 'ugrizy')

#### Have a look at the qualities we just extracted

Notice here that we have event more np.nan, because if either made is a non-detection the color is not defined

Note that we have still haven't accounted for galactic redenning. 

#### Now let's plot some of these colors

In [ ]:
plotting_functions.plot_color_scatter(colors_cModel, colors_psf, ['u-g', 'g-r', 'r-i', 'i-z', 'z-y'], mask=mask)
plotting_functions.plot_color_resid(colors_cModel, colors_psf, ['u-g', 'g-r', 'r-i', 'i-z', 'z-y'], mask=mask)

In [ ]:
plotting_functions.plot_color_scatter(colors_1p0, colors_psf, ['u-g', 'g-r', 'r-i', 'i-z', 'z-y'], mask=mask)
plotting_functions.plot_color_resid(colors_1p0, colors_psf, ['u-g', 'g-r', 'r-i', 'i-z', 'z-y'], mask=mask)

### Ok, let's summarize what we are seeing in terms of the comparisons between different version of the colors